### Data Cleaning

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent   # because notebook is inside /code
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.data_preprocessing import preprocess_data

df = preprocess_data("../data/Personal_Finance_Dataset.csv")
df.head()

,date,category,amount,type,signed_amount
0,2020-01-02,Food & Drink,1485.69,Expense,-1485.69
1,2020-01-02,Utilities,1475.58,Expense,-1475.58
2,2020-01-04,Rent,1185.08,Expense,-1185.08
3,2020-01-05,Investment,2291.00,Income,2291.00
4,2020-01-13,Food & Drink,1126.88,Expense,-1126.88


### Monthly Spending (Feature Engineering)

In [3]:
from src.monthly_spending import save_monthly_spending_table

monthly_table = save_monthly_spending_table(df, "../data/monthly_spending_table.csv")
monthly_table.head()

category,month,Entertainment,Food & Drink,Health & Fitness,Rent,Salary,Shopping,Travel,Utilities
0,2020-01,1914.85,5458.51,2370.34,3663.22,1077.09,1178.66,0.00,1475.58
1,2020-02,3199.52,1829.57,766.81,1222.17,294.31,1499.79,7272.28,1023.96
2,2020-03,1887.05,1353.18,881.82,6241.21,0.00,1929.08,486.51,802.96
3,2020-04,3821.84,1938.30,1603.18,0.00,2222.60,388.92,3195.04,3063.17
4,2020-05,0.00,2501.62,2893.98,3822.55,318.05,1318.17,4288.25,1703.51


### Trends and Volatility (Feature Engineering)

In [9]:
from src.trends import calculate_trends_and_volatility

final_behavioral_df = calculate_trends_and_volatility(monthly_table)
final_behavioral_df.to_csv("../data/trends_volatility.csv", index=False)

final_behavioral_df.head()

category,month,Entertainment,Food & Drink,Health & Fitness,Rent,Salary,Shopping,Travel,Utilities,total_expense,rolling_3m_expense,expense_mom_change,spending_volatility
0,2020-01,1914.85,5458.51,2370.34,3663.22,1077.09,1178.66,0.00,1475.58,17138.25,NaN,NaN,NaN
1,2020-02,3199.52,1829.57,766.81,1222.17,294.31,1499.79,7272.28,1023.96,17108.41,NaN,-0.001741,NaN
2,2020-03,1887.05,1353.18,881.82,6241.21,0.00,1929.08,486.51,802.96,13581.81,15942.823333,-0.206133,0.128255
3,2020-04,3821.84,1938.30,1603.18,0.00,2222.60,388.92,3195.04,3063.17,16233.05,15641.090000,0.195205,0.117403
4,2020-05,0.00,2501.62,2893.98,3822.55,318.05,1318.17,4288.25,1703.51,16846.13,15553.663333,0.037767,0.111547


### Frequency & Regularity (Feature Engineering)

In [5]:
from src.frequency_regularity import save_frequency_regularity

freq_reg_df = save_frequency_regularity(df, "../data/frequency_regularity.csv")
freq_reg_df.head()

,month,transaction_count,avg_inter_txn_gap,spending_dispersion,unique_categories_used
0,2020-01,15,2.071429,481.654677,7
1,2020-02,15,1.928571,614.515339,8
2,2020-03,15,1.500000,708.592010,7
3,2020-04,13,2.416667,595.698755,7
4,2020-05,21,1.500000,605.952522,7


### Habit-Based Indicators (Feature Engineering)

In [6]:
from src.habit_indicators import save_habit_indicators

habit_df = save_habit_indicators(
    monthly_df=final_behavioral_df,
    raw_df=df,
    output_path="../data/habit_indicators.csv"
)

habit_df.head()

,month,Entertainment,Food & Drink,Health & Fitness,Rent,Salary,Shopping,Travel,Utilities,total_expense,...,discretionary_total,essential_total,discretionary_ratio,total_income,monthly_savings,savings_rate,binge_spending_flag,binge_category,dominant_category,dominant_category_share
0,2020-01,1914.85,5458.51,2370.34,3663.22,1077.09,1178.66,0.00,1475.58,17138.25,...,3093.51,10597.31,0.291915,5578.0,-11560.25,-2.072472,0,None,Food & Drink,0.318499
1,2020-02,3199.52,1829.57,766.81,1222.17,294.31,1499.79,7272.28,1023.96,17108.41,...,11971.59,4075.70,2.937309,20070.0,2961.59,0.147563,1,Travel,Travel,0.425070
2,2020-03,1887.05,1353.18,881.82,6241.21,0.00,1929.08,486.51,802.96,13581.81,...,4302.64,8397.35,0.512381,3465.0,-10116.81,-2.919714,1,Rent,Rent,0.459527
3,2020-04,3821.84,1938.30,1603.18,0.00,2222.60,388.92,3195.04,3063.17,16233.05,...,7405.80,5001.47,1.480725,7370.0,-8863.05,-1.202585,0,None,Entertainment,0.235436
4,2020-05,0.00,2501.62,2893.98,3822.55,318.05,1318.17,4288.25,1703.51,16846.13,...,5606.42,8027.68,0.698386,6008.0,-10838.13,-1.803950,0,None,Travel,0.254554


### Overspending Target Creation

In [7]:
from src.target_creation import save_target_dataset

target_df = save_target_dataset(
    monthly_df=habit_df,
    output_path="../data/model_ready_with_target.csv"
)

target_df.head()

,month,Entertainment,Food & Drink,Health & Fitness,Rent,Salary,Shopping,Travel,Utilities,total_expense,...,discretionary_ratio,total_income,monthly_savings,savings_rate,binge_spending_flag,binge_category,dominant_category,dominant_category_share,overspend_current_month,overspend_target_t1
0,2020-01,1914.85,5458.51,2370.34,3663.22,1077.09,1178.66,0.00,1475.58,17138.25,...,0.291915,5578.0,-11560.25,-2.072472,0,None,Food & Drink,0.318499,1,0
1,2020-02,3199.52,1829.57,766.81,1222.17,294.31,1499.79,7272.28,1023.96,17108.41,...,2.937309,20070.0,2961.59,0.147563,1,Travel,Travel,0.425070,0,1
2,2020-03,1887.05,1353.18,881.82,6241.21,0.00,1929.08,486.51,802.96,13581.81,...,0.512381,3465.0,-10116.81,-2.919714,1,Rent,Rent,0.459527,1,1
3,2020-04,3821.84,1938.30,1603.18,0.00,2222.60,388.92,3195.04,3063.17,16233.05,...,1.480725,7370.0,-8863.05,-1.202585,0,None,Entertainment,0.235436,1,1
4,2020-05,0.00,2501.62,2893.98,3822.55,318.05,1318.17,4288.25,1703.51,16846.13,...,0.698386,6008.0,-10838.13,-1.803950,0,None,Travel,0.254554,1,1


### Modeling

In [8]:
import importlib
import src.modeling as modeling

importlib.reload(modeling)
from src.modeling import train_and_compare_models

results_df, fitted_models, split_data = train_and_compare_models(
    df=target_df,
    target_col="overspend_target_t1",
    test_size=0.2,
    random_state=42
)

results_df

,model,accuracy,f1_score,precision,recall,roc_auc,avg_precision
0,logistic_regression,0.750000,0.857143,0.750000,1.000000,0.444444,0.775253
1,random_forest,0.750000,0.857143,0.750000,1.000000,0.148148,0.626708
2,gradient_boosting,0.666667,0.800000,0.727273,0.888889,0.444444,0.729798
